In [1]:
#As usual, loading all libraries
!pip install -q langchain langchain-community transformers sentence-transformers faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [2]:
from science_rag import ScienceRAGAssistant #Loading the module including all necessary functions

In [4]:
#Creating an instance
# We must give the directory of the vector database, and we keep using the Qwen model
rag = ScienceRAGAssistant(faiss_path="faiss_Sci", model_name="Qwen/Qwen2.5-0.5B-Instruct")

/content/science_rag.py:7: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  self.embedding_model = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Error in void faiss::read_xb_vector(VectorT&, IOReader*) [with VectorT = MaybeOwnedVector<unsigned char>] at /project/third-party/faiss/faiss/impl/index_read.cpp:201: Error: 'ret == (size)' failed: read error in faiss_Sci/index.faiss: 12582867 != 20662272 (Success)

In [ ]:
#Running a simple test
answer= rag.ask(question="What approaches are used for image classification in this context?",
              k=3, allow_model_knowledge=False)

print(answer)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I don't know.


# Building a Simple Interface for the Science RAG Assistant

In this notebook, we build a small interface for our RAG assistant.

The interface will allow the user to:
- enter a science-related question
- choose how many chunks to retrieve
- decide whether the model can use its own knowledge
- receive the final answer

## Building UI using Gradio

Gradio is a Python library for quickly building simple web interfaces for models.

It allows us to connect inputs (e.g. text, sliders) to a function and display the output interactively.

In this workshop, we use it to turn our RAG pipeline into an easy-to-use app.

In [5]:
!pip install -q gradio langchain langchain-community transformers sentence-transformers faiss-gpu-cu12

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 7.9 MB/s eta 0:00:00


In [6]:
import gradio as gr

## Define a Wrapper Function

Gradio needs a function that takes the user inputs and returns the final answer.

In [7]:
def ask_science_assistant(question, k, allow_model_knowledge):
    if not question.strip():
        return "Please enter a question."

    answer = rag.ask(
        question=question,
        k=int(k),
        allow_model_knowledge=allow_model_knowledge
    )
    return answer

In [8]:
demo = gr.Interface(
    fn=ask_science_assistant,
    inputs=[
        gr.Textbox(lines=2, placeholder="Enter your science question here...", label="Question"),
        gr.Slider(1, 5, value=3, step=1, label="Number of retrieved chunks"),
        gr.Checkbox(value=False, label="Allow model to use its own knowledge")
    ],
    outputs=gr.Textbox(lines=8, label="Answer"),
    title="Science RAG Assistant",
    description="Ask a science-related question using a simple RAG pipeline.",
    flagging_mode="never"
)

In [9]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c1de1bec96cc9df70b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
